# Cost Functions and Unit Selection

From the target phonemes, a path needs to be found through the state transition network (sequence of database units), that minimizes a cost function.

![image.png](resources/UnitSelectionCosts.png)

## Target Cost

The target cost is the weighted sum of the differences between elements of the target and candidate feature vectors.

$$ C^{t}(t_i, u_i) = \sum_{j=1}^{p} w_j C^{t}_{j}(t_i, u_i) $$

Where $p$ ranges from 20 to 30 and is the number of sub-costs, or differences between the feature vectors.

## Concatenation Cost

Determined by the weight of $q$ concatenation sub-costs.

$$ C^c_{(u_{i-1}, u_i)} = \sum_{j=1}^q w^c_j C_j^{c}(u_{i-1}, u_i) $$

## Final Cost Function

$$ C(t_1^n, u_1^n) = \sum_{i=1}^{n} \sum_{j=1}^{p} w_j^tC_j^t(t_i, u_i^t) + \sum_{i=2}^{n} \sum_{j=1}^{q} w_j^cC_j^c(u_j, u_i) + C^s(S, u_1) + C^c(u_n, S) $$

Where $S$ is silence.

The units are then selected in order to minimize this cost function via Viterbi search.

$$ \bar{u}_1^n = \min_{u_1,\ldots,u_n} \mathcal{C}(t_1^n,u_1^n) $$

In [7]:
# For each file in "datasets/cmu_us_awb_arctic/lab"
# For each line in file 2:N-2 (1-indexed)
# Extract current phoneme, next phoneme, previous phoneme, duration.
from pathlib import Path

LABEL_DIR = 'datasets/cmu_us_awb_arctic/lab/'

def proc_line(line):
    t_end, _, ph = line.strip().split(' ')
    return float(t_end), ph

def ph_data_from_labels(label_dir):
    ph_feats = []
    src_dir = Path(label_dir)
    for i, file in enumerate(src_dir.iterdir()):
        with open(file, "r", encoding="utf-8") as cfile:
            lines = cfile.readlines()
            for i in range(2, len(lines) - 2):
                cd, cph = proc_line(lines[i])
                pd, pph = proc_line(lines[i - 1])
                _, nph = proc_line(lines[i + 1])
                ldata = {'cph' : cph, 'nph': nph, 'pph': pph, 'dur': cd - pd}
                ph_feats.append(ldata)
    return ph_feats

In [8]:
import numpy as np

def feat_from_ph_lut(lut_entry):
    is_vowel = [lut_entry['is_vowel']]
    is_voiced = [lut_entry['is_voiced']]

    manner_vec = [1 if lut_entry['manner'] == m else 0 for m in MANNERS]
    place_vec = [1 if lut_entry['place'] == p else 0 for p in PLACES]

    return np.array(is_vowel + is_voiced + manner_vec + place_vec, dtype=np.float32)

def feat_from_ph_data(ph_data):
    curr_ph_vec = [1 if ph_data['cph'] == cph else 0 for cph in PHONEME_FEATURES]
    next_ph_vec = [1 if ph_data['nph'] == cph else 0 for cph in PHONEME_FEATURES]
    prev_ph_vec = [1 if ph_data['pph'] == cph else 0 for cph in PHONEME_FEATURES]
    dur = [ph_data['dur']]
    return np.array(curr_ph_vec + next_ph_vec + prev_ph_vec + dur)

# Processing feature data into feature vectors
# Each phoneme from the labelled files produces a feature vector which is the sum of the label vector and the lut vector
def const_feature_vecs(phoneme_data, phoneme_lut):
    lut_feats = {}
    for key in phoneme_lut:
        lut_feats[key] = feat_from_ph_lut(phoneme_lut[key])
    
    feature_vecs = []
    for entry in phoneme_data:
        feat = feat_from_ph_data(entry)
        feature_vecs.append(np.concatenate([feat, lut_feats[entry['cph']]]))

    return feature_vecs

# Storing feature vectors offline
def serialize_features(feature_vecs):
    np.savez_compressed('datasets/cmu_us_awb_arctic/feats/featsa', *feature_vecs)


In [64]:
# Generate and serialize features

ph_data = ph_data_from_labels(LABEL_DIR)

serialize_features(const_feature_vecs(ph_data, PHONEME_FEATURES))

# Cost Functions Continued

Coming back to the cost functions as I've made a mistake in my perception. The weight and cost values are scalar, and there are $p$ of them. My thinking was rooted in neural networks where we have one cost function for the output of the network. In this application, we want to calculate costs and their weights more explicitly for the unique features. This is because the features vary greatly and the same cost function is not appropriately applied to all.

For the current feature set, this means that we have a cost and weight for each of: binary comparisons, each one-hot, and the duration.

I will keep this in mind when I go to train weights.

In [ ]:
import pandas as pd

def serialize_acoustic_features(frames):
    frames.to_parquet('datasets/cmu_us_awb_arctic/feats/acoustic_feats_a.parquet', engine='fastparquet', compression='snappy')
    

In [80]:

serialize_acoustic_features(const_acoustic_features('datasets/cmu_us_awb_arctic/'))